### Load libraries

In [0]:
%pip install databricks-feature-store

  Using cached databricks_feature_store-0.17.0-py3-none-any.whl.metadata (3.4 kB)
Using cached databricks_feature_store-0.17.0-py3-none-any.whl (4.2 kB)
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


### Define catalog as object 

In [0]:
# # Definice tvé struktury v Unity Catalogu
# catalog_name = "workspace"  # Tvůj zjištěný katalog
# schema_name = "default"    # Tvé schéma
# volume_name = "mlflow_tmp"

# Pokud chceš používat tu zkratku DA (jako v kurzech), udělej tohle:
class DA:
    catalog_name = "pipeline_new"
    schema_name = "default"
    volume_name = "mlflow_tmp"

print(DA.catalog_name)
print(DA.schema_name)
print(DA.volume_name)

pipeline_new
default
mlflow_tmp


### Load data

In [0]:
# Načtení tabulky přímo z katalogu
df_bronze = spark.table("pipeline_new.default.input")

# Zobrazení dat
display(df_bronze)

id,name,age,education,experience,department,is_manager,country,gender,salary,hobby
id_001,John,28,Bachelor,6,IT,0,USA,Male,65000,null
id_002,Alice,45,Master,21,Sales,1,Germany,Female,115000,null
id_003,Martin,null,Secondary,5,HR,0,Czechia,Male,38000,null
id_004,Chris,55,PhD,28,IT,1,USA,Male,145000,null
id_005,Emma,31,Bachelor,9,Marketing,0,Germany,Female,62000,null
id_006,Liam,25,Secondary,7,Sales,0,USA,Male,46000,null
id_006,Liam,25,Secondary,7,Sales,0,USA,Male,46000,null
id_007,Olivia,42,Master,18,IT,0,Germany,Female,98000,null
id_008,Noah,60,PhD,33,Sales,1,Czechia,Male,155000,null
id_009,Sophia,29,Bachelor,7,HR,0,Czechia,Female,55000,books


Databricks visualization. Run in Databricks to view.

###Data types

In [0]:
df_bronze.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: long (nullable = true)
 |-- education: string (nullable = true)
 |-- experience: long (nullable = true)
 |-- department: string (nullable = true)
 |-- is_manager: long (nullable = true)
 |-- country: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: long (nullable = true)
 |-- hobby: string (nullable = true)



In [0]:
from pyspark.sql.functions import col

# Takhle změníš typy u konkrétních sloupců
df_bronze = (df_bronze.withColumn("age", col("age").cast("int"))
            .withColumn("salary", col("salary").cast("double"))
            .withColumn("experience", col("experience").cast("int"))
            .withColumn("is_manager", col("is_manager").cast("int"))
)

# Kontrola
df_bronze.printSchema()

root
 |-- id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- education: string (nullable = true)
 |-- experience: integer (nullable = true)
 |-- department: string (nullable = true)
 |-- is_manager: integer (nullable = true)
 |-- country: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- hobby: string (nullable = true)



### Drop duplicates

In [0]:
# Odstraníme duplikáty
df_gold = df_bronze.dropDuplicates()

# Spočítáme řádky před
count_before = df_bronze.count()

# Spočítáme řádky po
count_after = df_gold.count()

print(f"Odstraněno {count_before - count_after} úplných duplikátů.")

Odstraněno 2 úplných duplikátů.


### Check NaN

In [0]:
from pyspark.sql.functions import col, when, count, lit, round

# 1. Spočítáme celkový počet řádků
total_rows = df_gold.count()

# 2. Spočítáme NULLy pro každý sloupec
null_counts = df_gold.select([
    count(when(col(c).isNull(), c)).alias(c) 
    for c in df_gold.columns
])

# 3. Převedeme sloupce na řádky (unpivot) a dopočítáme procenta
# Použijeme stack, abychom dostali název sloupce a jeho počet nulls pod sebe
stack_expr = ", ".join([f"'{c}', `{c}`" for c in df_bronze.columns])

missing_df = null_counts.selectExpr(f"stack({len(df_gold.columns)}, {stack_expr}) as (column_name, null_count)") \
    .withColumn("percentage_nan", round((col("null_count") / total_rows) * 100, 2)) \
    .select("column_name", "percentage_nan")

missing_df.show(truncate=False)

+-----------+--------------+
|column_name|percentage_nan|
+-----------+--------------+
|id         |0.0           |
|name       |0.0           |
|age        |13.33         |
|education  |0.0           |
|experience |0.0           |
|department |0.0           |
|is_manager |0.0           |
|country    |0.0           |
|gender     |0.0           |
|salary     |0.0           |
|hobby      |93.33         |
+-----------+--------------+



In [0]:
# Vyfiltrujeme sloupce s více než 60 % NaN a uložíme jejich názvy do seznamu
cols_to_drop = [row['column_name'] for row in missing_df.filter(col("percentage_nan") > 60).collect()]

print(f"Sloupce k odstranění (> 60% NaN): {cols_to_drop}")

Sloupce k odstranění (> 60% NaN): ['hobby']


In [0]:
# Odstranění sloupců z tvého hlavního DataFrame
print(f"Původní počet sloupců: {len(df_gold.columns)}")

df_gold = df_gold.drop(*cols_to_drop)

print(f"Nový počet sloupců: {len(df_gold.columns)}")

Původní počet sloupců: 11
Nový počet sloupců: 10


### Save golden table into Feature Store

In [0]:
from databricks.feature_store import FeatureStoreClient
fs = FeatureStoreClient()

In [0]:
# show available catalogs
display(spark.sql("SHOW CATALOGS"))

catalog
pipeline_new
samples
system


In [0]:
# Definuj název (v Unity Catalogu: catalog.schema.table_name)
feature_table_name = f"{DA.catalog_name}.{DA.schema_name}.salary_features"

# Vytvoření tabulky ve Feature Storu
fs.create_table(
    name=feature_table_name,
    primary_keys=["id"],  # Tvoje unikátní ID
    df=df_gold,        # Tvůj vyčištěný DataFrame (bez duplikátů a NaN sloupců)
    schema=df_gold.schema,
    description="Feature table contains all important features for model training."
)

2026/04/26 19:31:17 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['id'] of table 'pipeline_new.default.salary_features' to NOT NULL.
2026/04/26 19:31:19 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['id'] on table 'pipeline_new.default.salary_features'.
2026/04/26 19:31:26 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'pipeline_new.default.salary_features'.


<FeatureTable: name='pipeline_new.default.salary_features', table_id='0cd4682c-830c-4bb9-912d-921f0acf1ae9', description='Feature table contains all important features for model training.', primary_keys=['id'], partition_columns=[], features=['id',
 'name',
 'age',
 'education',
 'experience',
 'department',
 'is_manager',
 'country',
 'gender',
 'salary'], creation_timestamp=1777231877083, online_stores=[], notebook_producers=[], job_producers=[], table_data_sources=[], path_data_sources=[], custom_data_sources=[], timestamp_keys=[], tags={}>

### Pipeline

In [0]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.sql.functions import col, when
from pyspark.ml.feature import SQLTransformer

#### Pipeline - feature engineering

In [0]:

# 0. Ordinální kategorická proměnná
# Definuješ SQL výraz. 
# '__THIS__' v PySparku znamená "aktuální tabulka, která zrovna protéká pipelinou"

education_mapping = SQLTransformer(statement="""
    SELECT *,
    CASE 
        WHEN education = 'Primary' THEN 1
        WHEN education = 'Secondary' THEN 2
        WHEN education = 'Bachelor' THEN 3
        WHEN education = 'Master' THEN 4
        WHEN education = 'PhD' THEN 5
        ELSE 0 
    END as education_num
    FROM __THIS__
""")


# 1. Imputer na čísla
imputer = Imputer(
    inputCols=["age", "experience"], 
    outputCols=["age", "experience"], 
    strategy="median"
)


# 2. Indexer pro VŠECHNY kategorie (nominální i binární)
# Přidali jsme gender a is_manager, aby se s nimi Pipeline poprala sama
indexer = StringIndexer(
    inputCols=["country", "department", "gender"], 
    outputCols=["country_idx", "dept_idx", "gender_idx"],
    handleInvalid="keep"
)

# 3. OHE jen pro ty s více kategoriemi (Country, Dept)
ohe = OneHotEncoder(
    inputCols=["country_idx", "dept_idx", "gender_idx", "is_manager"], 
    outputCols=["country_ohe", "dept_ohe", "gender_ohe", "is_manager_ohe"],
    dropLast=True
)

# 4. Assembler - teď bere všechno dohromady
assembler = VectorAssembler(
    inputCols=[
        "age", 
        "experience", 
        "education_num", 
        "gender_ohe",  # OHE vektory
        "is_manager_ohe", # OHE vektory
        "country_ohe", # OHE vektory
        "dept_ohe"     # OHE vektory
    ], 
    outputCol="unscaled_features"
)

# 5. Scaler - Finální úprava měřítka
scaler = StandardScaler(inputCol="unscaled_features", outputCol="scaled_features")


# # 6. Model
# model = LinearRegression(
#     featuresCol="scaled_features", 
#     labelCol="salary", 
#     regParam=0.0
# )


#### Pipeline - define function

In [0]:
def build_pipeline(model):

    return Pipeline(stages=[
        education_mapping,
        imputer,
        indexer,
        ohe,
        assembler,
        scaler,
        model
    ])

#### Pipeline - define models

In [0]:
from pyspark.ml.regression import (
    LinearRegression,
    RandomForestRegressor,
    GBTRegressor
)

models = {
    "linear": LinearRegression(
        featuresCol="scaled_features",
        labelCol="salary",
        regParam=0.0
    ),

    "ridge": LinearRegression(
    featuresCol="scaled_features",
    labelCol="salary",
    regParam=0.1,
    elasticNetParam=0.0
    ),

    "lasso": LinearRegression(
    featuresCol="scaled_features",
    labelCol="salary",
    regParam=0.1,
    elasticNetParam=1.0
    ),

    "rf": RandomForestRegressor(
        featuresCol="scaled_features",
        labelCol="salary",
        numTrees=100
    ),

    "gbt": GBTRegressor(
        featuresCol="scaled_features",
        labelCol="salary",
        maxIter=50
    )
}

### Load features from FS and 

#### Using lookup

In [0]:
# from databricks.feature_store import FeatureLookup

# # 1. Definujeme "Levou stranu" (Base)
# base_df = spark.table("workspace.default.salary_features").select("id", "salary")

# # 2. Definujeme "Pravou stranu" (Lookup / Předpis na Join)
# feature_lookups = [
#     FeatureLookup(
#         table_name="workspace.default.salary_features",
#         feature_names=["age", "is_manager", "gender", "experience", "education", "country", "department"],
#         lookup_key="id"
#     )
# ]

# # 3. Provedeme ten "Join" skrze Feature Store
# training_set = fs.create_training_set(
#     df=base_df,
#     feature_lookups=feature_lookups,
#     label="salary",
#     exclude_columns=["id"]
# )

# # 4. Načteme výslednou spojenou tabulku
# loaded_fs_table = training_set.load_df()

# loaded_fs_table.display()

#### Not using lookup

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient
fe = FeatureEngineeringClient()

# Jednoduché načtení celé tabulky bez Lookupů
loaded_fs_table = fe.read_table(name=f"{DA.catalog_name}.{DA.schema_name}.salary_features")
loaded_fs_table.display(5)


id,name,age,education,experience,department,is_manager,country,gender,salary
id_008,Noah,60,PhD,33,Sales,1,Czechia,Male,155000.0
id_022,Daniel,40,Master,16,IT,0,Germany,Male,88000.0
id_020,Alexander,59,PhD,32,Sales,1,USA,Male,148000.0
id_028,Henry,47,Bachelor,25,HR,1,USA,Male,105000.0
id_019,Ava,43,Master,19,IT,0,Czechia,Female,92000.0
id_003,Martin,null,Secondary,5,HR,0,Czechia,Male,38000.0
id_030,William,56,PhD,29,Sales,1,Czechia,Male,149000.0
id_001,John,28,Bachelor,6,IT,0,USA,Male,65000.0
id_013,Charlotte,null,Bachelor,11,HR,0,USA,Female,64000.0
id_027,Luna,29,Primary,11,Sales,0,Germany,Female,39000.0


### Train/test split

In [0]:
# Rozdělení na trénovací (80 %) a testovací (20 %) sadu
# Seed zaručuje, že když kód pustíš znova, dostaneš stejné rozdělení

df_train, df_test = loaded_fs_table.randomSplit([0.8, 0.2], seed=42)

# Jen pro kontrolu, kolik tam toho máme
print(f"Počet řádků v trénovací sadě: {df_train.count()}")
print(f"Počet řádků v testovací sadě: {df_test.count()}")

Počet řádků v trénovací sadě: 23
Počet řádků v testovací sadě: 7


### ML flow

##### Enable writing ML flow into Unity catalog

In [0]:
import os
import mlflow

# Přepne MLflow na Unity Catalog a nastaví dočasnou složku
mlflow.set_registry_uri("databricks-uc")
os.environ['MLFLOW_DFS_TMP'] = f"/Volumes/{DA.catalog_name}/{DA.schema_name}/{DA.volume_name}"

##### Metrics

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator
import pyspark.sql.functions as F

# Standardní evaluátory
evaluator_rmse = RegressionEvaluator(labelCol="salary", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="salary", predictionCol="prediction", metricName="r2")
evaluator_mae = RegressionEvaluator(labelCol="salary", predictionCol="prediction", metricName="mae")

In [0]:
from mlflow.models.signature import infer_signature
import gc

results = []


for name, model in models.items():
    with mlflow.start_run(run_name=name):
        
        # 1. Training (Estimator v akci)
        pipeline = build_pipeline(model)

        pipeline_model = pipeline.fit(df_train)
        predictions = pipeline_model.transform(df_test)
        
        # 3. Calculation of metrics
        # Standardní metriky
        rmse = evaluator_rmse.evaluate(predictions)
        r2 = evaluator_r2.evaluate(predictions)
        mae = evaluator_mae.evaluate(predictions)
        
        # mape
        mape = predictions.withColumn("abs_perc_err", F.abs((F.col("salary") - F.col("prediction")) / F.col("salary"))) \
                        .select(F.mean("abs_perc_err") * 100).collect()[0][0]
    
        
        # SIGNATURE Z RAW DAT
        sample_df = df_test.limit(5)
        sample_predictions = pipeline_model.transform(sample_df)

        signature = infer_signature(
            sample_df.drop("salary"), 
            sample_predictions.select("prediction")
        )
        
        # 4. Logging 
        # 4.1. Logging Metrics 
        mlflow.log_metric("rmse", rmse)
        mlflow.log_metric("r2", r2)
        mlflow.log_metric("mae", mae)
        mlflow.log_metric("mape", mape)
        

        # 4.2. Logging pipeline 
        mlflow.spark.log_model(
            spark_model=pipeline_model,
            artifact_path="model",
            signature=signature,           
        )

        # 5. 🔥 append do results
        results.append({
            "model": name,
            "rmse": rmse,
            "r2": r2,
            "mae": mae,
            "mape": mape
        })
        
        print(f"{name}: RMSE: {rmse:.2f}, R2: {r2:.2f}, MAE: {mae:.2f}, MAPE: {mape:.2f}%")


        spark.catalog.clearCache()

print("Done! Check runs in Experiment!!!")

/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/26 19:32:35 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be time-consuming. To speed up, explicitly specify the conda_env or pip_requirements when cal

2026/04/26 19:33:08 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/755300380237305/57428211a0764543bed54e5d295c5c76/artifacts/model/sparkml, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 


linear: RMSE: 30139.58, R2: 0.60, MAE: 23062.70, MAPE: 40.09%


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/26 19:33:58 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be time-consuming. To speed up, explicitly specify the conda_env or pip_requirements when cal

2026/04/26 19:34:26 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/755300380237305/0b6f4bb1f45e464e8a793c0fe02ec3fe/artifacts/model/sparkml, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 


rf: RMSE: 13994.23, R2: 0.91, MAE: 11104.38, MAPE: 18.33%


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/04/26 19:36:02 INFO mlflow.spark: Inferring pip requirements by reloading the logged model from the databricks artifact repository, which can be time-consuming. To speed up, explicitly specify the conda_env or pip_requirements when cal

2026/04/26 19:36:30 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: dbfs:/databricks/mlflow-tracking/755300380237305/ca004412ef4c448fb318db779a5e5288/artifacts/model/sparkml, flavor: spark). Fall back to return ['pyspark==4.1.0']. Set logging level to DEBUG to see the full traceback. 


gbt: RMSE: 17184.43, R2: 0.87, MAE: 12345.01, MAPE: 14.75%
Done! Check runs in Experiment!!!


### Metrics

In [0]:
display(results)



mae,mape,model,r2,rmse
23062.698717168885,40.09022192857547,linear,0.5950572227447095,30139.576692144565
11104.380952380949,18.326816692446762,rf,0.9126993462602853,13994.231945201962
12345.008379083158,14.753481833329044,gbt,0.8683594200439106,17184.433243297073


### Charts

In [0]:
import matplotlib.pyplot as plt
import pandas as pd

# 1. Stáhneme data a seřadíme je podle skutečného platu
# Tím vytvoříme plynulou křivku pro srovnání
pdf = predictions.select("salary", "prediction").toPandas()
pdf = pdf.sort_values(by="salary").reset_index(drop=True)

# 2. Vykreslení
plt.figure(figsize=(12, 6))

# Skutečná data (Salary)
plt.plot(pdf.index, pdf["salary"], label="Skutečný plat", color="blue", linewidth=2)

# Predikce (Prediction)
plt.plot(pdf.index, pdf["prediction"], label="Predikce modelu", color="orange", linestyle="--", alpha=0.8)

# Formátování
plt.xlabel("Pořadí testovacích vzorků (seřazeno dle platu)")
plt.ylabel("Plat")
plt.title("Srovnání trendu: Skutečnost vs. Predikce")
plt.legend()
plt.grid(True, alpha=0.3)

plt.show()

# Inference

In [0]:
import mlflow

# ID z tvého screenshotu
run_id = "e40f28ad349a493c91306a84da8b5c65"
model_uri = f"runs:/{run_id}/model"

# 1. Načtení
loaded_model = mlflow.pyfunc.load_model(model_uri)

# 2. Příprava dat (bez salary, aby zmizel ten warning)
sample_data = df_test.limit(2).drop("salary").toPandas()

# 3. Predikce
inference = loaded_model.predict(sample_data)

# 4. Výsledek
print(inference)